# TerraSR — Google Colab Notebook

Terrain-conditioned satellite image super-resolution.  
Run cells top-to-bottom. Each section is idempotent — safe to re-run after a runtime reset.

**Before starting:** Runtime → Change runtime type → **T4 GPU** (or better).

---
| Section | What it does | Time (approx.) |
|---|---|---|
| 0 | GPU & environment check | < 1 min |
| 1 | Mount Google Drive | < 1 min |
| 2 | Clone repo + install deps | 3–5 min |
| 3 | Symlink Drive paths | < 1 min |
| 4 | Verify imports | < 1 min |
| 5 | Download subset dataset | 5–30 min |
| 6 | Run pipeline stages 2–6 | 15–60 min |
| 7 | Verify dataset | < 1 min |
| 8 | Single-batch smoke test | < 1 min |
| 9 | 5-epoch smoke training | 10–20 min |
| 10 | Checkpoint resume test | 2 min |
| 11 | Full training (all models) | hours — runs overnight |
| 12 | Evaluation | 5–10 min |
| 13 | Download results | < 1 min |

## Section 0 — GPU & Environment Check

In [ ]:
import subprocess, sys

# ── GPU info ──────────────────────────────────────────────────────────────────
print("=" * 60)
print("GPU INFO")
print("=" * 60)
try:
    result = subprocess.run(["nvidia-smi",
                             "--query-gpu=name,memory.total,driver_version",
                             "--format=csv,noheader"],
                            capture_output=True, text=True, check=True)
    name, vram, driver = [x.strip() for x in result.stdout.strip().split(",")]
    print(f"GPU      : {name}")
    print(f"VRAM     : {vram}")
    print(f"Driver   : {driver}")
except Exception as e:
    print(f"WARNING: nvidia-smi failed ({e}). Make sure a GPU runtime is selected.")

# ── PyTorch / CUDA ────────────────────────────────────────────────────────────
import torch
print()
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.version.cuda}")
print(f"GPU avail: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device   : {torch.cuda.get_device_name(0)}")
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM (PyTorch) : {total_gb:.1f} GB")
else:
    print()
    raise RuntimeError(
        "No CUDA GPU found. Go to Runtime → Change runtime type → T4 GPU, "
        "then reconnect and restart from Section 0."
    )

print()
print("Section 0 passed ✓")

## Section 1 — Mount Google Drive

A browser pop-up will ask you to authorise Colab to access your Drive.  
After mounting, the notebook creates `MyDrive/TerraSR-Colab/` with subdirectories for data, checkpoints, results, and logs.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = Path("/content/drive/MyDrive/TerraSR-Colab")

for subdir in ["data/raw/spacenet", "data/raw/maxar",
               "data/standardized", "data/patches",
               "data/pairs", "data/dataset", "data/cache/worldcover",
               "checkpoints", "results", "logs"]:
    (DRIVE_ROOT / subdir).mkdir(parents=True, exist_ok=True)

print("Drive mounted and TerraSR-Colab layout ready:")
print(f"  {DRIVE_ROOT}")
for p in sorted(DRIVE_ROOT.iterdir()):
    print(f"    {p.name}/")

print()
print("Section 1 passed ✓")

## Section 2 — Clone Repository & Install Dependencies

Clones the TerraSR repository from GitHub if not already present, then installs non-PyTorch dependencies from `colab/requirements-colab.txt`.  
**torch and torchvision are intentionally skipped** — Colab already provides a CUDA-enabled build.

VGG weights are pre-downloaded at the end of this section so the first training epoch does not stall on a 500 MB download.

In [ ]:
import subprocess, sys, os
from pathlib import Path

REPO_DIR = Path("/content/TerraSR")

# ── Clone ─────────────────────────────────────────────────────────────────────
if not REPO_DIR.exists():
    print("Cloning TerraSR repository...")
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/aaisha2/TerraSR.git", str(REPO_DIR)],
        check=True
    )
    print("Cloned.")
else:
    print(f"Repo already present at {REPO_DIR} — pulling latest...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
                   capture_output=True)
    print("Up to date.")

# ── Install deps (no torch/torchvision) ───────────────────────────────────────
req_file = REPO_DIR / "colab" / "requirements-colab.txt"
print(f"\nInstalling from {req_file.name} ...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(req_file)],
    check=True
)
print("Dependencies installed.")

# ── Add repo root to sys.path ─────────────────────────────────────────────────
repo_str = str(REPO_DIR)
if repo_str not in sys.path:
    sys.path.insert(0, repo_str)

# ── Pre-warm VGG weights ───────────────────────────────────────────────────────
# Download once here so training epochs don't stall later.
print("\nPre-downloading VGG16 weights (one-time, ~528 MB)...")
import torchvision.models as tvm
_ = tvm.vgg16(weights=tvm.VGG16_Weights.IMAGENET1K_V1)
print("VGG16 weights cached.")

print()
print("Section 2 passed ✓")

## Section 3 — Symlink Drive ↔ Repo Paths

Creates symlinks so `data/` and `checkpoints/` inside the cloned repo point to your Drive folders. This means every existing relative path in the research code (`data/dataset/train.csv`, `checkpoints/terrasr/last.pth`, etc.) resolves correctly — **no config files are changed**.

In [ ]:
import os
from pathlib import Path

REPO_DIR   = Path("/content/TerraSR")
DRIVE_ROOT = Path("/content/drive/MyDrive/TerraSR-Colab")

SYMLINKS = {
    REPO_DIR / "data":        DRIVE_ROOT / "data",
    REPO_DIR / "checkpoints": DRIVE_ROOT / "checkpoints",
}

os.chdir(REPO_DIR)   # all pipeline scripts use relative paths from here

for link, target in SYMLINKS.items():
    if link.is_symlink():
        current = os.readlink(link)
        if current == str(target):
            print(f"  ✓ {link.name}/ → (already linked)")
            continue
        print(f"  updating symlink: {link.name}/ was → {current}")
        link.unlink()
    elif link.exists():
        # Move any locally-accumulated content to Drive first, then symlink
        import shutil
        print(f"  moving existing {link.name}/ to Drive...")
        shutil.copytree(str(link), str(target), dirs_exist_ok=True)
        shutil.rmtree(str(link))
    link.symlink_to(target)
    print(f"  ✓ {link.name}/ → {target}")

# Sanity check
assert (REPO_DIR / "data").resolve() == DRIVE_ROOT / "data"
assert (REPO_DIR / "checkpoints").resolve() == DRIVE_ROOT / "checkpoints"

print()
print("Working directory:", os.getcwd())
print("Section 3 passed ✓")

## Section 4 — Verify Imports

Confirms every TerraSR module loads without errors.

In [ ]:
import sys
from pathlib import Path

REPO_DIR = Path("/content/TerraSR")
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

errors = []

checks = [
    ("torch",                             "import torch"),
    ("rasterio",                          "import rasterio"),
    ("timm",                              "import timm"),
    ("models (registry)",                 "import models"),
    ("models.terrasr_swinir",             "from models.terrasr_swinir import build_model"),
    ("models.swinir_baseline",            "from models.swinir_baseline import build_model"),
    ("models.srcnn_baseline",             "from models.srcnn_baseline import build_model"),
    ("models.srgan_baseline",             "from models.srgan_baseline import build_model"),
    ("TerrainAwareLoss",                  "from models.losses.terrain_aware_loss import TerrainAwareLoss"),
    ("TerraSRDataset",                    "from terrasr_data import TerraSRDataset, load_terrain_index"),
    ("train_utils",                       "from training.train_utils import get_device, save_checkpoint"),
]

for label, stmt in checks:
    try:
        exec(stmt)
        print(f"  ✓ {label}")
    except Exception as e:
        print(f"  ✗ {label}: {e}")
        errors.append(label)

if errors:
    raise ImportError(f"Failed imports: {errors}. Re-run Section 2 to fix dependencies.")

print()
print("Section 4 passed ✓")

## Section 5 — Download Subset Dataset

Downloads a small, representative subset to validate the end-to-end pipeline before committing to the full dataset.

**Edit the variables below to change which AOIs/events and how many files to download.**  
Set `SPACENET_MAX_FILES = None` and `MAXAR_MAX_FILES = None` to download everything.

In [ ]:
# ── Download configuration ────────────────────────────────────────────────────
# Phase 1 (smoke test): small subset for pipeline validation
# Phase 3 (full experiment): add more AOIs/events and set max_files to None

SPACENET_AOIS       = ["AOI_2_Vegas", "AOI_5_Khartoum"]  # add more for Phase 3
SPACENET_MAX_FILES  = 3    # scenes per AOI; None = all

MAXAR_EVENTS        = ["Brazil-Flooding-May24"]            # add more for Phase 3
MAXAR_MAX_FILES     = 2    # tiles per event; None = all

print("SpaceNet AOIs :", SPACENET_AOIS, f"(max {SPACENET_MAX_FILES} files each)")
print("Maxar events  :", MAXAR_EVENTS,  f"(max {MAXAR_MAX_FILES} tiles each)")

In [ ]:
import subprocess, sys
from pathlib import Path

REPO_DIR = Path("/content/TerraSR")

print("=" * 60)
print("Downloading SpaceNet PAN imagery")
print("=" * 60)

cmd = [sys.executable,
       str(REPO_DIR / "data_pipeline/01_download/download_spacenet.py"),
       "--config", str(REPO_DIR / "configs/datasets.yaml")]
for aoi in SPACENET_AOIS:
    cmd += ["--aoi", aoi]
if SPACENET_MAX_FILES is not None:
    cmd += ["--max-files", str(SPACENET_MAX_FILES)]

result = subprocess.run(cmd, cwd=REPO_DIR, capture_output=False)
if result.returncode != 0:
    raise RuntimeError("SpaceNet download failed — check error output above.")
print("\nSpaceNet download complete ✓")

In [ ]:
import subprocess, sys
from pathlib import Path

REPO_DIR = Path("/content/TerraSR")

print("=" * 60)
print("Downloading Maxar PAN tiles")
print("=" * 60)

cmd = [sys.executable,
       str(REPO_DIR / "data_pipeline/01_download/download_maxar.py"),
       "--config", str(REPO_DIR / "configs/datasets.yaml")]
for event in MAXAR_EVENTS:
    cmd += ["--event", event]
if MAXAR_MAX_FILES is not None:
    cmd += ["--max-files", str(MAXAR_MAX_FILES)]

result = subprocess.run(cmd, cwd=REPO_DIR, capture_output=False)
if result.returncode != 0:
    raise RuntimeError("Maxar download failed — check error output above.")
print("\nMaxar download complete ✓")

## Section 6 — Run Data Pipeline (Stages 2–6)

Each stage is run as a separate cell so you can see exactly what's happening and resume from any intermediate stage if something fails. All scripts are idempotent — re-running skips already-processed files.

In [ ]:
import subprocess, sys
from pathlib import Path

REPO_DIR = Path("/content/TerraSR")

def run_stage(label, cmd):
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")
    result = subprocess.run([str(c) for c in cmd], cwd=REPO_DIR)
    if result.returncode != 0:
        raise RuntimeError(f"Stage failed: {label}")
    print(f"  {label} — done ✓")

PY     = sys.executable
CFGS   = REPO_DIR / "configs"
DIRS   = {
    "raw":          REPO_DIR / "data/raw",
    "standardized": REPO_DIR / "data/standardized",
    "patches":      REPO_DIR / "data/patches",
    "pairs":        REPO_DIR / "data/pairs",
    "dataset":      REPO_DIR / "data/dataset",
}
print("Helpers loaded ✓")

In [ ]:
# Stage 2 — standardize scenes for each source present in data/raw/
import yaml
pipeline_cfg = yaml.safe_load((REPO_DIR / "configs/pipeline.yaml").read_text())

for source_name, src in pipeline_cfg["sources"].items():
    raw_dir = DIRS["raw"] / source_name
    if not raw_dir.exists():
        print(f"  [skip] {source_name}: not in data/raw/")
        continue
    cmd = [PY, REPO_DIR / "data_pipeline/02_standardize/standardize_scenes.py",
           "--in-dir", raw_dir,
           "--out-dir", DIRS["standardized"] / source_name,
           "--mode", src["mode"]]
    if src.get("recursive"):
        cmd.append("--recursive")
    run_stage(f"standardize/{source_name}", cmd)

In [ ]:
# Stage 3 — extract 256×256 patches
run_stage("patchify",
    [PY, REPO_DIR / "data_pipeline/03_patchify/tile_extractor.py",
     "--in-dir",  DIRS["standardized"], "--recursive",
     "--out-dir", DIRS["patches"],
     "--config",  CFGS / "patchify.yaml"])

# Stage 3 continued — filter low-content patches
run_stage("patch_filter",
    [PY, REPO_DIR / "data_pipeline/03_patchify/patch_filter.py",
     "--manifest", DIRS["patches"] / "patch_manifest.json",
     "--config",   CFGS / "patchify.yaml"])

In [ ]:
# Stage 4 — WorldCover terrain labeling
# ESA WorldCover tiles are cached in data/cache/worldcover/ on Drive after first download.
run_stage("worldcover_zonal_stats",
    [PY, REPO_DIR / "data_pipeline/04_labeling/worldcover_zonal_stats.py",
     "--manifest", DIRS["patches"] / "patch_manifest_filtered.json",
     "--config",   CFGS / "terrain_classes.yaml",
     "--only-kept"])

run_stage("assign_dominant_terrain",
    [PY, REPO_DIR / "data_pipeline/04_labeling/assign_dominant_terrain.py",
     "--manifest", DIRS["patches"] / "patch_manifest_zonal.json",
     "--config",   CFGS / "terrain_classes.yaml"])

In [ ]:
# Stage 5 — generate LR/HR GeoTIFF pairs via degradation pipeline
run_stage("make_lr_hr_pairs",
    [PY, REPO_DIR / "data_pipeline/05_degrade/make_lr_hr_pairs.py",
     "--manifest",    DIRS["patches"] / "patch_manifest_labeled.json",
     "--out-dir",     DIRS["pairs"],
     "--config",      CFGS / "degradation.yaml",
     "--only-labeled"])

In [ ]:
# Stage 6 — build unified manifest and produce train/val/test CSVs
run_stage("build_manifest",
    [PY, REPO_DIR / "data_pipeline/06_package/build_manifest.py",
     "--labeled",     DIRS["patches"] / "patch_manifest_labeled.json",
     "--degradation", DIRS["pairs"]   / "degradation_manifest.json",
     "--out-dir",     DIRS["dataset"]])

run_stage("split_train_val_test",
    [PY, REPO_DIR / "data_pipeline/06_package/split_train_val_test.py",
     "--manifest", DIRS["dataset"] / "dataset_manifest.parquet",
     "--config",   CFGS / "split.yaml"])

run_stage("dataset_stats",
    [PY, REPO_DIR / "data_pipeline/06_package/dataset_stats.py",
     "--manifest", DIRS["dataset"] / "dataset_manifest_split.parquet",
     "--config",   CFGS / "split.yaml"])

print("\nAll pipeline stages complete ✓")

## Section 7 — Verify Dataset

Confirms that the stage 6 manifests are readable, the terrain distribution looks sensible, and one batch loads correctly through the PyTorch Dataset.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import torch
from torch.utils.data import DataLoader

REPO_DIR = Path("/content/TerraSR")
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from terrasr_data import TerraSRDataset, load_terrain_index

terrain_index = load_terrain_index(REPO_DIR / "configs/terrain_classes.yaml")

for split_name, csv_path in [("train", "data/dataset/train.csv"),
                              ("val",   "data/dataset/val.csv"),
                              ("test",  "data/dataset/test.csv")]:
    p = REPO_DIR / csv_path
    if not p.exists():
        print(f"  {split_name}: {csv_path} not found — check stage 6 output")
        continue
    df = pd.read_csv(p)
    print(f"\n{split_name} ({len(df):,} patches):")
    if "terrain_label" in df.columns:
        print(df["terrain_label"].value_counts().to_string())

# Load one batch
print("\n--- Batch load test ---")
ds = TerraSRDataset(REPO_DIR / "data/dataset/train.csv",
                    terrain_index=terrain_index,
                    normalize="per_patch_max", augment=False)
loader = DataLoader(ds, batch_size=4, shuffle=False, num_workers=0)
lr, hr, terrain_idx, meta = next(iter(loader))
print(f"LR  shape : {tuple(lr.shape)}  dtype: {lr.dtype}  range: [{lr.min():.3f}, {lr.max():.3f}]")
print(f"HR  shape : {tuple(hr.shape)}  dtype: {hr.dtype}  range: [{hr.min():.3f}, {hr.max():.3f}]")
print(f"terrain_idx: {terrain_idx.tolist()}")
print(f"terrain_label: {meta['terrain_label']}")

print()
print("Section 7 passed ✓")

## Section 8 — Single-Batch Smoke Test

Builds TerraSR and runs one forward + backward pass without a full training loop. This confirms the GPU, model, and loss all work together before committing to any training time.

In [ ]:
import sys, yaml
from pathlib import Path
import torch

REPO_DIR = Path("/content/TerraSR")
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import models
from terrasr_data import load_terrain_index
from models.losses.terrain_aware_loss import TerrainAwareLoss
from training.train_utils import get_device

device = get_device()
print(f"Device: {device}")
torch.cuda.reset_peak_memory_stats()

# Build model
cfg = yaml.safe_load((REPO_DIR / "configs/train_terrasr.yaml").read_text())
model = models.build("terrasr", cfg["model"]).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"TerraSR params: {n_params/1e6:.2f}M")

# Build loss (perceptual disabled to avoid another VGG forward during smoke test)
terrain_index = load_terrain_index(REPO_DIR / "configs/terrain_classes.yaml")
loss_cfg = yaml.safe_load((REPO_DIR / "configs/terrain_aware_loss.yaml").read_text())
for w in loss_cfg["per_terrain"].values():
    w["perceptual"] = 0.0
loss_cfg["default"]["perceptual"] = 0.0
criterion = TerrainAwareLoss(loss_cfg, terrain_index).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)

# Synthetic batch (no dataset needed for the smoke test)
BATCH = 2
LR_SIZE, HR_SIZE = 64, 128   # 2× scale factor
lr_t = torch.rand(BATCH, 1, LR_SIZE, LR_SIZE, device=device)
hr_t = torch.rand(BATCH, 1, HR_SIZE, HR_SIZE, device=device)
terrain_idx = torch.tensor([0, 2], device=device)   # Urban, Forest

# Forward + backward
optimizer.zero_grad()
sr = model(lr_t, terrain_idx)
loss, components = criterion(sr, hr_t, terrain_idx)
loss.backward()
optimizer.step()

mem_mb = torch.cuda.max_memory_allocated() / 1e6
print(f"SR output shape : {tuple(sr.shape)}")
print(f"Loss            : {loss.item():.4f}")
print(f"Components      : {components}")
print(f"Peak GPU memory : {mem_mb:.0f} MB")

print()
print("Section 8 passed ✓")

## Section 9 — Smoke Training (5 Epochs)

Runs a short training run with conservative settings to confirm the full loop works — data loading, loss computation, checkpointing to Drive — before committing to the overnight run.

**Checkpoint will appear at:** `MyDrive/TerraSR-Colab/checkpoints/smoke_terrasr/last.pth`

In [ ]:
import subprocess, sys
from pathlib import Path

REPO_DIR = Path("/content/TerraSR")

print("=" * 60)
print("Smoke training — TerraSR, 5 epochs")
print("=" * 60)

cmd = [
    sys.executable, str(REPO_DIR / "training/train_terrasr.py"),
    "--config", str(REPO_DIR / "configs/train_terrasr.yaml"),
    "--override",
        "train.epochs=5",
        "train.batch_size=4",
        "train.num_workers=2",
        "train.out_dir=checkpoints/smoke_terrasr",
        "loss.disable_perceptual=true",   # skip VGG for speed
]
result = subprocess.run(cmd, cwd=REPO_DIR)
if result.returncode != 0:
    raise RuntimeError("Smoke training failed — check output above.")

# Verify checkpoint exists on Drive
ckpt = REPO_DIR / "checkpoints/smoke_terrasr/last.pth"
assert ckpt.exists(), f"Expected checkpoint at {ckpt}"
size_mb = ckpt.stat().st_size / 1e6
print(f"\nCheckpoint saved: {ckpt} ({size_mb:.1f} MB)")
print("Section 9 passed ✓")

## Section 10 — Checkpoint Resume Test

Simulates a runtime reset by re-running the same training command (no `--fresh`).  
You should see `resuming from epoch 6` in the output, confirming the crash-recovery path works.

In [ ]:
import subprocess, sys
from pathlib import Path

REPO_DIR = Path("/content/TerraSR")

print("=" * 60)
print("Resume test — should print 'resuming from epoch 6'")
print("=" * 60)

cmd = [
    sys.executable, str(REPO_DIR / "training/train_terrasr.py"),
    "--config", str(REPO_DIR / "configs/train_terrasr.yaml"),
    "--override",
        "train.epochs=5",           # already done; should print 'nothing to do'
        "train.batch_size=4",
        "train.num_workers=2",
        "train.out_dir=checkpoints/smoke_terrasr",
        "loss.disable_perceptual=true",
]
result = subprocess.run(cmd, cwd=REPO_DIR)
if result.returncode != 0:
    raise RuntimeError("Resume test failed — check output above.")

print()
print("Section 10 passed ✓")
print("(If you saw 'already trained 5 epochs; nothing to do' that is also correct —")
print(" it means resume logic detected the completed checkpoint and skipped gracefully.")

## Section 11 — Full Training

Trains all four models (SRCNN, SRGAN, SwinIR, TerraSR) at full config. This is the overnight run.

**Batch size guidance:**
- T4 (15 GB) → `batch_size=8`
- L4 (22 GB) → `batch_size=12`
- A100 (40 GB) → `batch_size=16` (original research setting)

Checkpoints are saved to Drive after every epoch — if the runtime terminates, re-run Sections 0–4 then re-run Section 11; training resumes from the last completed epoch automatically.

In [ ]:
import torch

# Detect available VRAM and suggest a batch size
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU VRAM: {total_gb:.1f} GB")

if total_gb >= 38:
    BATCH_SIZE = 16
elif total_gb >= 20:
    BATCH_SIZE = 12
else:
    BATCH_SIZE = 8

print(f"Auto-selected batch_size: {BATCH_SIZE}")
print("(Edit BATCH_SIZE below to override if you get OOM errors.)")

In [ ]:
import subprocess, sys
from pathlib import Path

REPO_DIR = Path("/content/TerraSR")

print("=" * 60)
print("Training SRCNN baseline")
print("=" * 60)

result = subprocess.run([
    sys.executable, str(REPO_DIR / "training/train_baseline.py"),
    "--config", str(REPO_DIR / "configs/train_baseline.yaml"),
    "--override",
        "model.name=srcnn",
        f"train.batch_size={BATCH_SIZE}",
        "train.num_workers=2",
        "train.out_dir=checkpoints/srcnn",
], cwd=REPO_DIR)
if result.returncode != 0:
    raise RuntimeError("SRCNN training failed.")
print("SRCNN done ✓")

In [ ]:
import subprocess, sys
from pathlib import Path

REPO_DIR = Path("/content/TerraSR")

print("=" * 60)
print("Training SRGAN baseline")
print("=" * 60)

result = subprocess.run([
    sys.executable, str(REPO_DIR / "training/train_baseline.py"),
    "--config", str(REPO_DIR / "configs/train_baseline.yaml"),
    "--override",
        "model.name=srgan",
        f"train.batch_size={BATCH_SIZE}",
        "train.num_workers=2",
        "train.out_dir=checkpoints/srgan",
], cwd=REPO_DIR)
if result.returncode != 0:
    raise RuntimeError("SRGAN training failed.")
print("SRGAN done ✓")

In [ ]:
import subprocess, sys
from pathlib import Path

REPO_DIR = Path("/content/TerraSR")

print("=" * 60)
print("Training SwinIR baseline")
print("=" * 60)

result = subprocess.run([
    sys.executable, str(REPO_DIR / "training/train_baseline.py"),
    "--config", str(REPO_DIR / "configs/train_baseline.yaml"),
    "--override",
        "model.name=swinir",
        f"train.batch_size={BATCH_SIZE}",
        "train.num_workers=2",
        "train.out_dir=checkpoints/swinir",
], cwd=REPO_DIR)
if result.returncode != 0:
    raise RuntimeError("SwinIR training failed.")
print("SwinIR done ✓")

In [ ]:
import subprocess, sys
from pathlib import Path

REPO_DIR = Path("/content/TerraSR")

print("=" * 60)
print("Training TerraSR (terrain-conditioned model)")
print("=" * 60)

result = subprocess.run([
    sys.executable, str(REPO_DIR / "training/train_terrasr.py"),
    "--config", str(REPO_DIR / "configs/train_terrasr.yaml"),
    "--override",
        f"train.batch_size={BATCH_SIZE}",
        "train.num_workers=2",
        # leave loss.disable_perceptual unset (false) — use full terrain-aware loss
], cwd=REPO_DIR)
if result.returncode != 0:
    raise RuntimeError("TerraSR training failed.")
print("TerraSR done ✓")

## Section 12 — Evaluation

Runs the full evaluation suite against all four trained models using the held-out test split:
- Global PSNR/SSIM table (with bicubic baseline)
- Per-terrain PSNR breakdown
- Results report (Markdown + `.docx`)

Outputs are saved to `MyDrive/TerraSR-Colab/results/`.

In [ ]:
import subprocess, sys
from pathlib import Path

REPO_DIR = Path("/content/TerraSR")

CKPTS = [
    str(REPO_DIR / "checkpoints/srcnn/best.pth"),
    str(REPO_DIR / "checkpoints/srgan/best.pth"),
    str(REPO_DIR / "checkpoints/swinir/best.pth"),
    str(REPO_DIR / "checkpoints/terrasr/best.pth"),
]

# Check all checkpoints exist
missing = [c for c in CKPTS if not Path(c).exists()]
if missing:
    print("WARNING: some checkpoints not found — evaluation will run on available ones:")
    for m in missing:
        print(f"  MISSING: {m}")
    CKPTS = [c for c in CKPTS if Path(c).exists()]

TEST_CSV   = str(REPO_DIR / "data/dataset/test.csv")
SPLIT_MF   = str(REPO_DIR / "data/dataset/dataset_manifest_split.csv")
RESULTS    = REPO_DIR / "data/dataset/results_report"   # -> Drive via symlink

def run_eval(label, cmd):
    print(f"\n--- {label} ---")
    r = subprocess.run([str(c) for c in cmd], cwd=REPO_DIR)
    if r.returncode != 0:
        print(f"  WARNING: {label} exited with {r.returncode}")
    else:
        print(f"  {label} — done ✓")

PY = sys.executable

run_eval("PSNR/SSIM table",
    [PY, REPO_DIR / "evaluation/eval_psnr_ssim.py",
     "--test-csv", TEST_CSV, "--with-bicubic",
     "--checkpoints", *CKPTS])

run_eval("Per-terrain PSNR",
    [PY, REPO_DIR / "evaluation/eval_per_terrain.py",
     "--test-csv", TEST_CSV,
     "--checkpoints", *CKPTS])

run_eval("Results report",
    [PY, REPO_DIR / "evaluation/make_results_report.py",
     "--test-csv",       TEST_CSV,
     "--split-manifest", SPLIT_MF,
     "--out",            str(RESULTS),
     "--checkpoints",    *CKPTS])

print()
print("Section 12 complete ✓")
print(f"Results saved to: {RESULTS}  (also accessible in Drive)")

## Section 13 — Download Results

Downloads the results report directly to your local machine. The files also remain on Google Drive.

In [ ]:
from google.colab import files
from pathlib import Path

REPO_DIR = Path("/content/TerraSR")
RESULTS  = REPO_DIR / "data/dataset/results_report"

to_download = list(RESULTS.glob("*.docx")) + list(RESULTS.glob("*.md")) + \
              list(RESULTS.glob("*.csv"))

if not to_download:
    print(f"No result files found in {RESULTS}. Run Section 12 first.")
else:
    print(f"Downloading {len(to_download)} result file(s)...")
    for f in to_download:
        print(f"  {f.name}")
        files.download(str(f))
    print()
    print("All results downloaded to your local machine ✓")
    print(f"They are also available on Drive at:")
    print(f"  MyDrive/TerraSR-Colab/data/dataset/results_report/")